## Dataset Creation notebooks

## Imports

In [2]:
from collections import Counter
from random import randint

from datasets import load_dataset
from datasets import Audio, DatasetDict, concatenate_datasets, Dataset
import torch
import librosa
import numpy as np
from IPython.lib.display import Audio as AudioDisplay

SEED = 42
NUM_PROC = 24
SAMPLING_RATE = 16000
CHUNK_DURATION = 0.5
BATCH_SIZE = 32

/home/pierre/Documents/Projects/PST4/AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [3]:
def display_ds(ds):
    print(f"Size of splits: train={len(ds['train'])}, val={len(ds['val'])}")
    print(f"Labels {set(ds['train']['label'])}")
    i = randint(0, len(ds["train"])-1)
    print(f"Label: {ds['train'][i]['label']}")
    display(AudioDisplay(ds["train"][i]["audio"]["array"], rate=ds["train"][i]["audio"]["sampling_rate"]))

## Load dataset and select n%

In [4]:
base_ds = load_dataset("n1coc4cola/maotouying")
base_ds_train = base_ds["train"]
n_instance = 1
base_ds_train_shuffled = base_ds_train.shuffle(seed=SEED).select(range(int(n_instance * len(base_ds_train))))

## Split in train/test

In [5]:
train_test = base_ds_train_shuffled.train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
matoying_ds = DatasetDict({
    "train": train_test["train"],
    "val": train_test["test"],
})
for split in matoying_ds.keys():
    matoying_ds[split] = matoying_ds[split].cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
display_ds(matoying_ds)

Size of splits: train=168771, val=42193
Labels {0, 1}
Label: 0


## Add other datasets

In [7]:
ds_all_tests = load_dataset("Hibou-Foundation/all_tests_ds_3")

## Split train/test

In [8]:
dataset = DatasetDict({
    "train": Dataset.from_dict({}),
    "val": Dataset.from_dict({}),
})

# For each split of all_tests_ds_3, we split it into train/test and concatenate with the current dataset
for split in ds_all_tests.keys():
    split_ds = ds_all_tests[split]
    split_ds = split_ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
    try:
        split_ds_train_test = split_ds.train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
    except ValueError:
        # If there is only one label in the split, we cannot stratify, so we do a simple train_test_split without stratification
        split_ds_train_test = split_ds.train_test_split(test_size=0.2, seed=SEED)

    dataset["train"] = concatenate_datasets([dataset["train"], split_ds_train_test["train"]])
    dataset["val"] = concatenate_datasets([dataset["val"], split_ds_train_test["test"]])
display_ds(dataset)

Size of splits: train=10387, val=2600
Labels {0, 1}
Label: 1


In [9]:
concatenated_dataset = DatasetDict({
    "train": concatenate_datasets([dataset["train"], matoying_ds["train"]]),
    "val": concatenate_datasets([dataset["val"], matoying_ds["val"]]),
})
concatenated_dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'label'],
        num_rows: 179158
    })
    val: Dataset({
        features: ['audio', 'label'],
        num_rows: 44793
    })
})

## Split un chunks of n seconds

In [10]:
def split_audio_into_chunks(audio_array, chunk_duration=CHUNK_DURATION, sampling_rate=SAMPLING_RATE):
    samples_per_chunk = int(chunk_duration * sampling_rate)
    num_chunks = audio_array.shape[-1] // samples_per_chunk
    # Only split into full chunks, no padding
    chunks = [audio_array[i * samples_per_chunk:(i + 1) * samples_per_chunk]
              for i in range(num_chunks)]

    return chunks

def chunk_audio_batch(batch):
    # Process by batch to allow multi processing
    all_audios = []
    all_sampling_rates = []
    all_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        audio_array = audio["array"]
        sampling_rate = audio["sampling_rate"]
        audio_array = torch.tensor(audio_array).float()
        chunks = split_audio_into_chunks(audio_array)

        all_audios.extend([chunk.numpy() for chunk in chunks])
        all_sampling_rates.extend([sampling_rate] * len(chunks))
        all_labels.extend([label] * len(chunks))

    return {
        "audio": all_audios,
        "label": all_labels,
    }


chunked_dataset = DatasetDict()
for split in dataset.keys():
    print(f"Chunking split: {split}, original length: {len(concatenated_dataset[split])}")
    chunked_split = concatenated_dataset[split].map(
        chunk_audio_batch,
        batched=True,
        num_proc=NUM_PROC,
        batch_size=BATCH_SIZE,
        remove_columns=concatenated_dataset[split].column_names,
    )
    print(f"Generated split {split} length, {len(chunked_split)}")
    chunked_dataset[split] = chunked_split


Chunking split: train, original length: 179158
Generated split train length, 636820
Chunking split: val, original length: 44793
Generated split val length, 158828


In [13]:
# chunked_dataset.save_to_disk("ds_1_MAT-AllTEST_chunked")

Saving the dataset (11/11 shards): 100%|██████████| 158828/158828 [00:18<00:00, 8647.19 examples/s] 


## Convert in melspectograms

In [13]:
def convert_to_mel_spectrogram(batch):
    spectograms = []
    labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        mel = librosa.feature.melspectrogram(
            y=np.asarray(audio),
            sr=16000,
            n_fft=1025,
            hop_length=256,
            n_mels=128,
            fmin=20,
            fmax=8000,
            power=2.0
        )
        # Convert to log scale (dB)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        # Normalize
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
        # Convert to torch tensor: [1, n_mels, time]
        mel_db = torch.tensor(mel_db).unsqueeze(0)

        spectograms.append(mel_db)
        labels.append(label)

    return {
        "audio": spectograms,
        "raw_audio": batch["audio"],
        "label": labels,
    }

spectogram_dataset = DatasetDict()
for split in chunked_dataset.keys():
    spectogram_split = chunked_dataset[split].map(
        convert_to_mel_spectrogram,
        batched=True,
        num_proc=10,
        batch_size=10,
        remove_columns=chunked_dataset[split].column_names,
    )
    spectogram_dataset[split] = spectogram_split

spectogram_dataset


Map (num_proc=10): 100%|██████████| 158828/158828 [14:24<00:00, 183.79 examples/s]


DatasetDict({
    train: Dataset({
        features: ['audio', 'label'],
        num_rows: 636820
    })
    val: Dataset({
        features: ['audio', 'label'],
        num_rows: 158828
    })
})

In [14]:
spectogram_dataset.save_to_disk("ds_1_MAT-AllTEST_melspectogram")

Saving the dataset (11/11 shards): 100%|██████████| 158828/158828 [00:17<00:00, 9146.31 examples/s] 
